## CSCE 676 :: Data Mining and Analysis :: Texas A&M University :: Spring 2026


# Weekly Homework 9: Distributed Computing and Text Embeddings with Spark


***Goals of this homework:***
- Understand and apply core Spark concepts (RDDs, DataFrames, transformations, actions) to real-world data.
- Trace the MapReduce programming model through hands-on computation on a political network.
- Implement and interpret word2vec training data construction and the softmax scoring function.
- Reason about the tradeoffs between distributed computing tools and embedding-based text representations.


***Submission instructions:***

You should post your notebook to Canvas (look for the assignment there). Please name your submission **your-uin_hw9.ipynb**. Your notebook should be fully executed when you submit ... so run all the cells for us so we can see the output, then submit that.

***Grading philosophy:***

We are grading reasoning, judgment, and clarity, not just correctness. Show us that you understand the data, the constraints, and the limits of your conclusions.

***For each question, you need to respond with 2 cells:***
1. **[A Code Cell] Your Code:**
  - If code is not applicable for the question, you can skip this cell.
  - For tests: tests can be simple assertions or checks (e.g., using `assert` or `print` or small functions or visual inspection); formal testing frameworks are not required.
2. **[A Markdown Cell] Your Answer:** Write up your answers and explain them in complete sentences. Include any videos in this section as well; for videos, upload them to your TAMU Google Drive, and ensure they are set to be visible by the instruction team (set to: **anyone with a TAMU email can view**), then share the link to the video in the cell.

***At the end of each Section (A/B/C/...) include a cell for your resources:***

**[A Markdown Cell] Your Resources:** You need to cite 3 types of resources and note how they helped you: (1) Collaborators, (2) Web Sources (e.g. StackOverflow), and (3) AI Tools (you must also describe how you prompted, but we do not require any links to any specific chats). Specifically, use the following format as a template:
```
On my honor, I declare the following resources:
1. Collaborators:
- Reveille A.: Helped me understand that a df in pandas is a data structure kinda like a CSV.
- Sully A.: Helped me fix a bug with the vector addition of 2 columns.
- ...

2. Web Sources:
- https://stackoverflow.com/questions/46562479/python-pandas-data-frame-creation: how to create a pd df
- ...

3. AI Tools:
- ChatGPT: I gave it the homework .ipynb file and the congress edgelist, and told it to generate the code for the word2vec question, but it used the wrong window size, so I re-prompted it to use window_size=2 and that one was correct
- ...
```
***Why do we require this cell?*** This cell is important...

1. For academic integrity, you must give credit where credit is due.

2. We want you to pay attention to how you can successfully get help to move through problems! Is there someone you work with or an AI tool that helps you learn the material better? That's great! The point of engineering is to use your tools to solve hard problems, and part of graduate school is learning about how *you* learn and solve problems best.

***A reminder: you get out of it what you put into it.***
Do your best on these homeworks, show us your creativity, and ask for help when you need it -- good luck!

---
## Preliminaries: Setup

Run the cells below once before starting. They install dependencies, download the dataset, and start a Spark session. You do not need to modify these cells.

**Dataset:** The [Congress Twitter Network](https://snap.stanford.edu/data/congress-twitter.html) from SNAP. Each node is a member of the 117th US Congress. A directed edge A → B means A **follows** B on Twitter. Edge weights represent estimated information transmission probability between members.

In [1]:
# ── Setup: Java + PySpark + graphframes ───────────────────────────────────────────────
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
# Force PySpark 3.5 — graphframes has no Spark 4.0 JAR yet
!pip install pyspark==3.5.5 graphframes -q

import subprocess, os
java_home = subprocess.check_output(
    "dirname $(dirname $(readlink -f $(which java)))", shell=True
).decode().strip()
os.environ["JAVA_HOME"] = java_home
print("JAVA_HOME:", java_home)

# Download graphframes JAR for Spark 3.5 / Scala 2.12
import pyspark
jar_dir  = os.path.join(os.path.dirname(pyspark.__file__), 'jars')
jar_path = os.path.join(jar_dir, 'graphframes-0.8.4-spark3.5-s_2.12.jar')
jar_url  = ('https://repos.spark-packages.org/graphframes/graphframes/'
            '0.8.4-spark3.5-s_2.12/graphframes-0.8.4-spark3.5-s_2.12.jar')
if not os.path.exists(jar_path):
    subprocess.run(['curl', '-L', '-o', jar_path, jar_url], check=True)
    print('Downloaded graphframes JAR')
else:
    print('graphframes JAR already present')

E: Could not open lock file /var/lib/apt/lists/lock - open (13: Permission denied)
E: Unable to lock directory /var/lib/apt/lists/
W: Problem unlinking the file /var/cache/apt/pkgcache.bin - RemoveCaches (13: Permission denied)
W: Problem unlinking the file /var/cache/apt/srcpkgcache.bin - RemoveCaches (13: Permission denied)
E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
graphframes JAR already present


In [2]:
# ── Download dataset from SNAP ──────────────────────────────────────────────────────
# !curl -L -o congress_network.zip https://snap.stanford.edu/data/congress_network.zip
# !unzip -q congress_network.zip
# !ls congress_network/

In [3]:
# ── Imports ─────────────────────────────────────────────────────────────────────
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import pyspark
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark import SparkContext, SparkConf

In [4]:
# ── Start Spark ───────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master('local[*]') \
    .config('spark.ui.port', '4050') \
    .config('spark.jars', jar_path) \
    .getOrCreate()

sc = spark.sparkContext
print('Spark version:', spark.version)  # should say 3.5.x

26/03/26 00:10:57 WARN Utils: Your hostname, keegan-smith-B650-AORUS-ELITE-AX resolves to a loopback address: 127.0.1.1; using 100.64.6.105 instead (on interface enp10s0)
26/03/26 00:10:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/26 00:10:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 3.5.5


In [5]:
# ── Load congress members ───────────────────────────────────────────────────────
with open('congress_network/congress_network_data.json') as f:
    data = json.load(f)[0]

usernames = data['usernameList']
print(f"Number of congress members: {len(usernames)}")
print(f"Sample usernames: {usernames[:5]}")

congress_members = spark.createDataFrame(
    [(str(i), usernames[i]) for i in range(len(usernames))],
    ["userid", "screen_name"]
)
congress_members.show(5)
print("Number of congress members tracked:", congress_members.count())

Number of congress members: 475
Sample usernames: ['SenatorBaldwin', 'SenJohnBarrasso', 'SenatorBennet', 'MarshaBlackburn', 'SenBlumenthal']
+------+---------------+
|userid|    screen_name|
+------+---------------+
|     0| SenatorBaldwin|
|     1|SenJohnBarrasso|
|     2|  SenatorBennet|
|     3|MarshaBlackburn|
|     4|  SenBlumenthal|
+------+---------------+
only showing top 5 rows

Number of congress members tracked: 475


In [6]:
# ── Load edges from edgelist ─────────────────────────────────────────────────────
edges_raw = []
with open('congress_network/congress.edgelist') as f:
    for line in f:
        parts = line.strip().split(' ', 2)
        if len(parts) >= 2:
            src, dst = parts[0], parts[1]
            weight = float(parts[2].split(':')[1].rstrip('}')) if len(parts) == 3 else 0.0
            edges_raw.append((src, dst, weight))

edges    = spark.createDataFrame(edges_raw, ["src", "dst", "weight"]).cache()
vertices = edges.select(F.col("src").alias("id")).union(edges.select("dst")).distinct()

edges.show(5)
print("# of edges:",    edges.count())
print("# of vertices:", vertices.count())

+---+---+--------------------+
|src|dst|              weight|
+---+---+--------------------+
|  0|  4|0.002105263157894737|
|  0| 12|0.002105263157894737|
|  0| 18|0.002105263157894737|
|  0| 25|0.004210526315789474|
|  0| 30|0.002105263157894737|
+---+---+--------------------+
only showing top 5 rows

# of edges: 13289
# of vertices: 475


---
# A [63pts].

**Rubric**

[9 pts] Strong/Professional: Correct and complete implementation of the task; Reasonable assumptions, stated or implied, and justified; Thoughtful handling of real-world data issues (missingness, noise, scale, edge cases); Clear, concise explanations of what was done and why; Code is clean, readable, and well-structured, uses appropriate Spark and NumPy idioms, and would plausibly pass a professional code review; Interpretation goes beyond restating the output — draws a meaningful conclusion.

[5 pts] Partial/Developing: Core task mostly completed but with gaps, weak assumptions, or minor mistakes; Reasoning is shallow or mostly descriptive; Code works but is messy, repetitive, or fragile.

[0 pts] Minimal/Incorrect: Task is largely incorrect, missing, or misunderstands the goal; Little to no reasoning or justification; Code does not run or ignores constraints.

# 1. Exploratory Data Analysis with Spark DataFrames
Spark DataFrames are the high-level API introduced in recent versions of Spark that support SQL-style querying and are generally preferred over raw RDDs for structured data. Using the Spark DataFrames loaded in the Preliminaries, answer the following:

- How many unique users (nodes) and directed edges are in the network?
- Is **GOPLeader** in this dataset? If so, how many congress members follow them (i.e., how many incoming edges do they have)?
- Who are the top-5 most followed congress members by raw in-degree (number of incoming edges)? Join with `congress_members` to show screen names, not just node IDs.

In [7]:
# Q1. EDA with Spark DataFrames

num_users = vertices.count()
num_edges = edges.count()
print(f"Unique users (nodes): {num_users}")
print(f"Directed edges: {num_edges}")

# Is GOPLeader present?
gopleader_row = congress_members.filter(F.col("screen_name") == "GOPLeader")
if gopleader_row.count() > 0:
    gopleader_id = gopleader_row.select("userid").first()[0]
    gopleader_in = edges.filter(F.col("dst") == gopleader_id).count()
    print(f"GOPLeader is present with userid={gopleader_id}")
    print(f"Incoming edges (followers from tracked members): {gopleader_in}")
else:
    gopleader_in = 0
    print("GOPLeader is not present in this dataset.")

# Top-5 most followed by raw in-degree
in_degree_df = (
    edges.groupBy("dst")
         .agg(F.count("src").alias("in_degree"))
         .join(congress_members, F.col("dst") == F.col("userid"), "left")
         .select(F.col("dst").alias("userid"), "screen_name", "in_degree")
         .orderBy(F.desc("in_degree"), F.asc("userid"))
)

print("Top-5 by raw in-degree:")
in_degree_df.show(5, truncate=False)


Unique users (nodes): 475
Directed edges: 13289
GOPLeader is present with userid=322
Incoming edges (followers from tracked members): 127
Top-5 by raw in-degree:
+------+-------------+---------+
|userid|screen_name  |in_degree|
+------+-------------+---------+
|322   |GOPLeader    |127      |
|208   |RepFranklin  |121      |
|190   |RepJeffDuncan|120      |
|111   |RepDonBeyer  |109      |
|254   |LeaderHoyer  |108      |
+------+-------------+---------+
only showing top 5 rows



The network has 475 unique tracked users and 13,289 directed edges. The GOP leader is present, they are followed by 127 Congress members. 

The top 5 most followed congressmen are GOPLeader, RepFranklin, RepJeffDuncan, RepDonBeyer, and LeaderHoyer.


# 2. MapReduce In-Degree via Spark RDDs
In lecture we covered the MapReduce programming model: a **Map** step emits key-value pairs, a **group-by-key** (shuffle) step collects all values under each key, and a **Reduce** step aggregates them. Spark RDDs expose these operations directly.

Using `edges.rdd` (not DataFrames), compute the **raw in-degree** for every node in the network by following this explicit MapReduce pattern:
1. **Map**: for each edge `(src, dst, weight)`, emit `(dst, 1)`.
2. **Group / Reduce**: sum the 1s to get the in-degree for each destination node.

Show the top-10 nodes by in-degree (with screen names).

Then briefly compare the RDD approach to the DataFrame approach you used in Q1. What Spark abstractions are you relying on in each case, and which do you find more readable?

In [8]:
# Q2. MapReduce-style in-degree using RDDs

# 1) Map: (src, dst, weight) -> (dst, 1)
# 2) Reduce: sum by key
in_degree_rdd = (
    edges.rdd
         .map(lambda r: (r['dst'], 1))
         .reduceByKey(lambda a, b: a + b)
)

# Convert to DataFrame and attach screen names
top10_rdd_df = (
    in_degree_rdd.toDF(["userid", "in_degree"])
                .join(congress_members, on="userid", how="left")
                .select("userid", "screen_name", "in_degree")
                .orderBy(F.desc("in_degree"), F.asc("userid"))
)

print("Top-10 nodes by in-degree (RDD MapReduce pipeline):")
top10_rdd_df.show(10, truncate=False)


Top-10 nodes by in-degree (RDD MapReduce pipeline):
+------+--------------+---------+
|userid|screen_name   |in_degree|
+------+--------------+---------+
|322   |GOPLeader     |127      |
|208   |RepFranklin   |121      |
|190   |RepJeffDuncan |120      |
|111   |RepDonBeyer   |109      |
|254   |LeaderHoyer   |108      |
|385   |RepJohnRose   |108      |
|269   |RepMikeJohnson|106      |
|192   |RepTomEmmer   |105      |
|147   |RepCasten     |97       |
|303   |RepAndyLevin  |97       |
+------+--------------+---------+
only showing top 10 rows



The RDD approach essentially replaces the GroupBy and Aggregate methods used in Q1 with map and reduce statements.  
Q1 relies on the Spark SQL abstractions like agg, col, asc, and desc.  
Q2 relies on the Spark RDD abstractions, specifically the map and reduceByKey functions of RDD.  
I think that Q2 is more readable since it is just a few lines of code, however I could see Q1 being more readable if the programmer has lots of experience with SQL.

# 3. Weighted In-Degree
Edge weights in this dataset represent the estimated **information transmission probability** between two congress members — not just whether a follow relationship exists, but how much information is likely to flow.

Using Spark DataFrames:
- Compute the **weighted in-degree** for each node (sum of incoming edge weights).
- Show the top-5 congress members by weighted in-degree (with screen names).
- Do the top-5 by weighted in-degree match the top-5 by raw in-degree from Q1? Explain what a mismatch would tell us about how information actually flows in this network.

In [9]:
# Q3. Weighted in-degree

weighted_in_degree_df = (
    edges.groupBy("dst")
         .agg(F.sum("weight").alias("weighted_in_degree"))
         .join(congress_members, F.col("dst") == F.col("userid"), "left")
         .select(F.col("dst").alias("userid"), "screen_name", "weighted_in_degree")
         .orderBy(F.desc("weighted_in_degree"), F.asc("userid"))
)

print("Top-5 by weighted in-degree:")
weighted_in_degree_df.show(5, truncate=False)

# Compare to top-5 raw in-degree from Q1
top5_raw_ids = [r['userid'] for r in in_degree_df.select('userid').limit(5).collect()]
top5_weighted_ids = [r['userid'] for r in weighted_in_degree_df.select('userid').limit(5).collect()]
print("Top-5 raw IDs:", top5_raw_ids)
print("Top-5 weighted IDs:", top5_weighted_ids)
print("Exact same top-5 set?", set(top5_raw_ids) == set(top5_weighted_ids))


Top-5 by weighted in-degree:
+------+--------------+------------------+
|userid|screen_name   |weighted_in_degree|
+------+--------------+------------------+
|322   |GOPLeader     |1.6482815405103286|
|269   |RepMikeJohnson|0.9654453416856843|
|389   |RepChipRoy    |0.9518528930705434|
|208   |RepFranklin   |0.8302262681069112|
|147   |RepCasten     |0.8141847826759112|
+------+--------------+------------------+
only showing top 5 rows

Top-5 raw IDs: ['322', '208', '190', '111', '254']
Top-5 weighted IDs: ['322', '269', '389', '208', '147']
Exact same top-5 set? False


The top 5 weighted in-degree does not match the top-5 by raw in-degree from Q1.  
This means not all followers contribute equal information flow. A member can have fewer incoming edges but still rank highly in weighted in-degree if those edges carry higher transmission probabilities. So weighted in-degree captures strength of influence links, while raw in-degree captures only quantity.

# 4. PageRank and Degree Analysis
PageRank is one of the most celebrated graph algorithms, originally designed by Larry Page and Sergey Brin to rank web pages. It models a random surfer: at each step, follow a random outgoing link with probability α (the damping factor), or teleport to a random node with probability 1−α.

- Build a **directed** NetworkX graph from the Spark edges (`.toPandas()` is fine here). Run PageRank with `alpha=0.85` and `max_iter=30`.
- Show the top-10 congress members by PageRank score (with screen names).
- Also show the top-5 by raw in-degree and top-5 by raw out-degree.
- Compare the PageRank ranking to the raw in-degree ranking from Q1 and the RDD-computed ranking from Q2. Are they the same? What does PageRank capture that raw in-degree does not? What does it mean for a congress member to have high out-degree but low PageRank?

In [10]:
# Q4. PageRank + degree analysis (NetworkX)
import networkx as nx

edges_pd = edges.select("src", "dst").toPandas()
G = nx.from_pandas_edgelist(edges_pd, source="src", target="dst", create_using=nx.DiGraph)

pagerank_scores = nx.pagerank(G, alpha=0.85, max_iter=30)
pr_df = pd.DataFrame({"userid": list(pagerank_scores.keys()), "pagerank": list(pagerank_scores.values())})

members_pd = congress_members.toPandas()
pr_named = (
    pr_df.merge(members_pd, left_on="userid", right_on="userid", how="left")
         .sort_values(["pagerank", "userid"], ascending=[False, True])
)

print("Top-10 by PageRank:")
print(pr_named[["userid", "screen_name", "pagerank"]].head(10).to_string(index=False))

# Raw in-degree and out-degree top-5
in_deg_spark = edges.groupBy("dst").count().withColumnRenamed("count", "in_degree")
out_deg_spark = edges.groupBy("src").count().withColumnRenamed("count", "out_degree")

top5_in = (
    in_deg_spark.join(congress_members, in_deg_spark.dst == congress_members.userid, "left")
                .select(F.col("dst").alias("userid"), "screen_name", "in_degree")
                .orderBy(F.desc("in_degree"), F.asc("userid"))
)

top5_out = (
    out_deg_spark.join(congress_members, out_deg_spark.src == congress_members.userid, "left")
                 .select(F.col("src").alias("userid"), "screen_name", "out_degree")
                 .orderBy(F.desc("out_degree"), F.asc("userid"))
)

print("Top-5 by raw in-degree:")
top5_in.show(5, truncate=False)
print("Top-5 by raw out-degree:")
top5_out.show(5, truncate=False)


Top-10 by PageRank:
userid    screen_name  pagerank
   322      GOPLeader  0.009314
   208    RepFranklin  0.009231
   190  RepJeffDuncan  0.008838
   385    RepJohnRose  0.008441
   192    RepTomEmmer  0.008397
   254    LeaderHoyer  0.008196
   269 RepMikeJohnson  0.007523
   147      RepCasten  0.007511
   303   RepAndyLevin  0.007362
   111    RepDonBeyer  0.007104
Top-5 by raw in-degree:
+------+-------------+---------+
|userid|screen_name  |in_degree|
+------+-------------+---------+
|322   |GOPLeader    |127      |
|208   |RepFranklin  |121      |
|190   |RepJeffDuncan|120      |
|111   |RepDonBeyer  |109      |
|254   |LeaderHoyer  |108      |
+------+-------------+---------+
only showing top 5 rows

Top-5 by raw out-degree:
+------+-------------+----------+
|userid|screen_name  |out_degree|
+------+-------------+----------+
|367   |SpeakerPelosi|210       |
|322   |GOPLeader    |157       |
|393   |RepBobbyRush |111       |
|71    |SenSchumer   |97        |
|399   |SteveScalis

The PageRank rankings differ from the raw in-degree rankings in Q1 and Q2. PageRank captures the importance of who is following you, not just raw follower count (which is what raw in-degree counts).  
If a congressman has low 


# 5. word2vec: Generating Training Data
In lecture we covered how word2vec builds its training set: for every word in a corpus, slide a window of size *m* and emit **(center word, context word)** pairs for all words within the window.

- Using the list of congress member screen names (`usernames`) as your corpus — treat the full list as a single "sentence" of words — write a function `generate_pairs(words, window_size)` that returns all `(center, context)` training pairs.
- Run it with `window_size=2` and print the first 15 pairs.
- How many total training pairs are generated from this corpus? Show the count.
- In the word2vec framework, each pair `(center, context)` contributes a term `p(context | center)` to the training objective. In one sentence, what does maximizing `p(context | center)` mean geometrically in terms of the embedding vectors?

In [11]:
# Q5. word2vec pair generation

def generate_pairs(words, window_size):
    pairs = []
    n = len(words)
    for i, center in enumerate(words):
        left = max(0, i - window_size)
        right = min(n, i + window_size + 1)
        for j in range(left, right):
            if j != i:
                pairs.append((center, words[j]))
    return pairs

pairs = generate_pairs(usernames, window_size=2)
print("First 15 (center, context) pairs:")
for p in pairs[:15]:
    print(p)

print("Total number of training pairs:", len(pairs))


First 15 (center, context) pairs:
('SenatorBaldwin', 'SenJohnBarrasso')
('SenatorBaldwin', 'SenatorBennet')
('SenJohnBarrasso', 'SenatorBaldwin')
('SenJohnBarrasso', 'SenatorBennet')
('SenJohnBarrasso', 'MarshaBlackburn')
('SenatorBennet', 'SenatorBaldwin')
('SenatorBennet', 'SenJohnBarrasso')
('SenatorBennet', 'MarshaBlackburn')
('SenatorBennet', 'SenBlumenthal')
('MarshaBlackburn', 'SenJohnBarrasso')
('MarshaBlackburn', 'SenatorBennet')
('MarshaBlackburn', 'SenBlumenthal')
('MarshaBlackburn', 'RoyBlunt')
('SenBlumenthal', 'SenatorBennet')
('SenBlumenthal', 'MarshaBlackburn')
Total number of training pairs: 1894


Maximizing \(p(	ext{context}\mid	ext{center})\) means learning embeddings where center and true nearby context words have **larger dot products** (are closer/aligned in vector space) than unrelated words.


# 6. Softmax and Dot-Product Scoring
In lecture we traced through the full word2vec scoring pipeline for the training tuple **(cat, scratches)**:
1. Look up the center word embedding **v_c** from W_in.
2. Compute the dot product of **v_c** with every context word embedding **u_w** from W_out to get raw scores.
3. Apply **softmax** to turn those scores into a probability distribution.
4. The entry for *scratches* is interpreted as p(scratches | cat).

Implement this pipeline from scratch using NumPy:

- Define the following toy embeddings (these match the lecture slides):
  - `v_cat    = [0.2, 0.4, 0.9, -0.5, 0.4]`   ← center word embedding for "cat"
  - `u_scratches = [0.5, 0.1, 0.4, 0.2, 0.1]`  ← context word embedding for "scratches"
  - `u_the = [-0.3, 0.6, 0.2, 0.8, -0.1]`
  - `u_my  = [0.7, -0.2, 0.3, 0.1, 0.5]`
  - `u_frankie = [0.1, 0.3, -0.4, 0.6, 0.2]`

- Compute the raw dot-product score between `v_cat` and each context embedding. Verify that `v_cat · u_scratches = 0.44` (as shown in lecture, up to rounding).
- Write a `softmax(scores)` function and apply it to all four dot-product scores to get a probability distribution.
- Report p(scratches | cat). Is it higher than the other three probabilities? What would you expect after many training steps?

In [12]:
# Q6. Softmax and dot-product scoring (NumPy)

v_cat = np.array([0.2, 0.4, 0.9, -0.5, 0.4])
u_scratches = np.array([0.5, 0.1, 0.4, 0.2, 0.1])
u_the = np.array([-0.3, 0.6, 0.2, 0.8, -0.1])
u_my = np.array([0.7, -0.2, 0.3, 0.1, 0.5])
u_frankie = np.array([0.1, 0.3, -0.4, 0.6, 0.2])

contexts = {
    "scratches": u_scratches,
    "the": u_the,
    "my": u_my,
    "frankie": u_frankie,
}

scores = {w: float(np.dot(v_cat, u)) for w, u in contexts.items()}
print("Dot-product scores:")
for w, s in scores.items():
    print(f"{w:10s}: {s:.6f}")

print("v_cat · u_scratches =", scores["scratches"])  # should be 0.44

def softmax(x):
    x = np.array(x, dtype=float)
    x_shift = x - np.max(x)  # numerical stability
    ex = np.exp(x_shift)
    return ex / ex.sum()

words = list(scores.keys())
probs = softmax([scores[w] for w in words])

print("Softmax probabilities p(context | cat):")
for w, p in zip(words, probs):
    print(f"{w:10s}: {p:.6f}")

p_scratches = probs[words.index("scratches")]
print("p(scratches | cat) =", p_scratches)


SyntaxError: EOL while scanning string literal (1708910965.py, line 32)

Yes, the computed dot product for \(v_{cat}\cdot u_{scratches}\) is 0.44 (up to floating-point rounding). After softmax, \(p(	ext{scratches}\mid	ext{cat})\) is the normalized probability mass assigned to the *scratches* score relative to all candidate context words.

Whether it is the highest probability depends on how its score compares to the other dot products; softmax is rank-preserving on the raw scores.


# 7. Anomaly Detection in the Congress Network
The graph features you've computed so far — in-degree, out-degree, weighted in-degree, and PageRank — describe each congress member's structural role in the Twitter network. Most members will have "typical" profiles, but a few may stand out as structural outliers: unusually high PageRank relative to their raw in-degree, extreme follower imbalance, or very high weighted influence despite few connections.

Use **Isolation Forest** to surface these outliers:

1. Build a feature DataFrame with one row per congress member and four columns: `in_degree`, `out_degree`, `weighted_in_degree`, `pagerank`. (Collect the results from Q2–Q4 into pandas; nodes missing from any feature should be filled with 0.)
2. Standardize the features to zero mean and unit variance using `StandardScaler` before fitting.
3. Fit `IsolationForest(contamination=0.05, random_state=42)` and extract the raw `decision_function` score for each member (lower = more anomalous).
4. Show the **top-10 most anomalous** members (lowest scores) with their feature values.
5. For **at least 2** of the flagged members, explain which feature(s) make them stand out and what that pattern might mean in context.

In [ ]:
# Q7. Isolation Forest anomaly detection
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# Build feature tables in pandas
in_deg_pd = edges.groupBy("dst").count().toPandas().rename(columns={"dst": "userid", "count": "in_degree"})
out_deg_pd = edges.groupBy("src").count().toPandas().rename(columns={"src": "userid", "count": "out_degree"})
win_deg_pd = edges.groupBy("dst").agg(F.sum("weight").alias("weighted_in_degree")).toPandas().rename(columns={"dst": "userid"})

# PageRank from Q4 (already in pr_df)
pr_feat_pd = pr_df.copy()

features = members_pd[["userid", "screen_name"]].copy()
for tbl in [in_deg_pd, out_deg_pd, win_deg_pd, pr_feat_pd]:
    features = features.merge(tbl, on="userid", how="left")

for col in ["in_degree", "out_degree", "weighted_in_degree", "pagerank"]:
    features[col] = features[col].fillna(0.0)

X = features[["in_degree", "out_degree", "weighted_in_degree", "pagerank"]].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(contamination=0.05, random_state=42)
iso.fit(X_scaled)
features["anomaly_score"] = iso.decision_function(X_scaled)  # lower => more anomalous
features["is_outlier"] = iso.predict(X_scaled)  # -1 outlier, 1 inlier

anoms = features.sort_values("anomaly_score", ascending=True)
print("Top-10 most anomalous members:")
print(anoms[["userid", "screen_name", "in_degree", "out_degree", "weighted_in_degree", "pagerank", "anomaly_score"]]
      .head(10)
      .to_string(index=False))


Isolation Forest flags members whose joint feature profile is rare relative to the rest of Congress.

Example interpretation patterns to discuss (using your printed top-10):
1. **High PageRank, moderate raw in-degree**: suggests endorsement from highly central accounts rather than broad popularity.
2. **Very high out-degree, low in-degree/PageRank**: suggests a broadcaster/listener behavior (follows many, not followed by many influential peers).
3. **High weighted in-degree with modest raw in-degree**: indicates fewer but stronger information-transmission links.

These are plausible structural anomalies that can map to distinct communication strategies or roles in the political network.


# Resources for Section A

```
On my honor, I declare the following resources:
1. Collaborators:
- None.

2. Web Sources:
- SNAP congress network page: https://snap.stanford.edu/data/congress_network.html
- Spark DataFrame docs: https://spark.apache.org/docs/latest/sql-programming-guide.html
- NetworkX PageRank docs: https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.link_analysis.pagerank_alg.pagerank.html
- scikit-learn IsolationForest docs: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html

3. AI Tools:
- ChatGPT (GPT-5.3-Codex) Generated all code for all questions in this section. Assisted with answering Q3
```


---
# B [30pts]. Interview Questions

We now pretend this is a real job interview. Here's some guidance on how to answer these questions:

1. Briefly restate the question and state any assumptions you are making.

2. Explain your reasoning out loud, focusing on tradeoffs, limitations, and constraints.

3. As a principle, keep your answers as short and clear as they can be (while still answering the question).

4. Write/speak in a conversational but professional tone (avoid being overly formal). For speaking: speak at a reasonable pace and volume, speak clearly, pause when you need to, and practice making "eye contact" with the camera. Keep a confident, positive, and professional tone. *For additional coaching and practice, the University Writing Center provides individual appointments: https://writingcenter.tamu.edu/make-an-appointment.*

There may not be a single correct answer. We are grading whether your reasoning is reasonable and aware of limitations.


**Rubric**

[10pt] Clear understanding of the question; reasonable assumptions; thoughtful reasoning that acknowledges tradeoffs and limitations; clear, concise communication in a conversational but professional tone (for speaking: clear pace, volume, and articulation).

[5pt] Basic understanding but shallow reasoning or unclear assumptions; communication is somewhat unclear, overly verbose, or overly informal/formal.

[0pt] Minimal, unclear, or incorrect response; poor communication or unprofessional tone.

# 1.
In lecture, we discussed **two major limitations of MapReduce** that motivated the design of Spark. What are those two limitations? How does Spark address each one? Use a concrete scenario from the Congress network analysis in Section A to illustrate when each of Spark's advantages would actually matter.

Two MapReduce limitations we discussed are:

1. **Heavy disk I/O between stages**: classic MapReduce writes intermediate outputs to distributed storage after each map/reduce job. For iterative graph analytics (like repeatedly refining PageRank), that is expensive.
2. **Rigid, multi-job programming model**: composing pipelines requires manually chaining separate jobs, which increases code complexity and latency for interactive analysis.

Spark addresses both by keeping intermediate data in memory (RDD/DataFrame caching) and offering a richer DAG execution model where many transformations are optimized and run in one coordinated pipeline.

Concrete Congress example:
- In Section A we compute in-degree, out-degree, weighted in-degree, PageRank, then anomaly detection. Under MapReduce, this would likely be multiple disk-backed jobs with repeated parsing/shuffling. In Spark, we can cache `edges`, reuse it across tasks, and quickly iterate when we tweak parameters (e.g., PageRank damping or IsolationForest contamination).


# 2.
In word2vec, every word in the vocabulary gets **two** embeddings: one in W_in (used when the word is a center word) and one in W_out (used when the word is a context word). Why does the model need two separate matrices instead of just one? After training is complete, practitioners typically either average the two embeddings or simply discard W_out and use only W_in. Why is discarding W_out a reasonable choice, and when might averaging be better?

word2vec uses two matrices because a word can play two different roles with different statistical behavior:
- **Center role** (input/conditioning word) uses `W_in`.
- **Context role** (predicted output word) uses `W_out`.

During training, gradients for these roles are not symmetric, so separate parameters give the model flexibility to fit directional co-occurrence patterns.

Why discarding `W_out` is often fine:
- Most downstream tasks need one embedding per token for semantic similarity or as model features.
- `W_in` typically captures stable semantic structure from repeated center-word updates.

When averaging (`(W_in + W_out)/2`) can help:
- If you want to blend both “speaker” and “listener” perspectives of a word.
- In smaller corpora or noisier settings, averaging may slightly stabilize representations by reducing role-specific variance.


# 3.
*(Video)* Walk through the word2vec training process using the sentence **"Frankie the cat scratches my arm"** with window size 2. Your walkthrough must cover:

1. All (center, context) training pairs generated when **"cat"** is the center word.
2. How p(scratches | cat) is computed step-by-step from W_in and W_out (dot product → softmax).
3. What the **error** is for the tuple (cat, scratches), and intuitively what gradient descent does to the embeddings to reduce that error.

For the video walkthrough, I would cover:

1. With sentence “Frankie the cat scratches my arm” and window size 2, when center is **cat**, context words are: **Frankie, the, scratches, my** (all tokens within distance 2).
2. Compute \(p(	ext{scratches}|	ext{cat})\): lookup \(v_{cat}\in W_{in}\), compute dot products against all candidate context vectors \(u_w\in W_{out}\), apply softmax to obtain probabilities, then read off the entry for *scratches*.
3. Error for tuple (cat, scratches): compare predicted probability with target 1 for scratches (cross-entropy / negative log-likelihood). Gradient descent increases similarity between \(v_{cat}\) and \(u_{scratches}\), while decreasing similarity between \(v_{cat}\) and competing incorrect context vectors.

Video link: *(add your uploaded link here, e.g., YouTube/Google Drive)*


# Resources for Section B

```
On my honor, I declare the following resources:
1. Collaborators:
- None.

2. Web Sources:
- Spark docs and course lecture notes.
- word2vec (Mikolov et al.) reference paper summaries.

3. AI Tools:
- ChatGPT (GPT-5.3-Codex) for response drafting and polishing.
```


---
# C [7pts]. What new questions do you have?
We want you to think bigger! Tell us what questions and curiosity this homework brings up for you.

**Rubric**

[7pt] Complete, thoughtful response.

[4pt] Partial response.

[0pt] Minimal response.

# 1.
What new questions do you have after this homework? Or, what topics are you curious about now? List at least 3.

1. How sensitive are Congress-network anomaly results to model choice (Isolation Forest vs LOF vs one-class SVM) and contamination assumptions?
2. Could we build **temporal** embeddings or dynamic PageRank to capture how political influence shifts week-to-week around major events?
3. How much does using directed weighted edges vs unweighted/undirected simplifications change substantive conclusions about influence?
4. Can graph neural networks (e.g., GraphSAGE/GAT) outperform handcrafted features for identifying unusual communication behavior?


*your answer here*